# Aadhaar Layout Detector — Colab Training (YOLOv8)

Trains the **layout/field** detector (aadhaar_number, dob, gender, name, address) — NOT the forgery model.

**Before running:**
1. Locally: `cd ml_service/data && zip -r aadhaar_yolo.zip aadhaar_yolo`
2. Upload `aadhaar_yolo.zip` to your Google Drive (root of MyDrive).
3. Runtime → Change runtime type → **T4 GPU**.

Output: `runs/detect/aadhaar_layout/weights/best.pt` (downloaded in the last cell).

In [ ]:
!pip install -q ultralytics
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Mount Drive and unzip the dataset prepared locally by prepare_aadhaar_yolo.py
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/aadhaar_yolo.zip -d /content/

In [ ]:
# data.yaml has a Windows absolute path baked in -> repoint it at the Colab location
import re, pathlib
p = pathlib.Path('/content/aadhaar_yolo/data.yaml')
p.write_text(re.sub(r'^path:.*', 'path: /content/aadhaar_yolo', p.read_text(), flags=re.M))
print(p.read_text())

In [ ]:
# Train. T4 (16GB) handles batch=32 easily; bump to yolov8s.pt at batch=16 for more accuracy.
!yolo detect train \
    data=/content/aadhaar_yolo/data.yaml \
    model=yolov8n.pt epochs=80 imgsz=640 batch=32 device=0 \
    cache=ram patience=20 name=aadhaar_layout

In [ ]:
# Download the trained field detector
from google.colab import files
files.download('runs/detect/aadhaar_layout/weights/best.pt')